<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
<font color="#76b900">
<h1 style="line-height: 1.4;"><b>Rapid Application Development<br>using Large Language Models</b></h1>
<h2><b>Notebook 7.6:</b> LangGraph</h2></font>
<br>

**Congratulations On (Almost) Finishing The Course!** We hope you've enjoyed the journey and have gained valuable skills to create advanced language model applications. 
- **In Notebook 8**, you'll be able to put those skills to the test to make an integrated system that straddles several domains.
- **In this notebook,** we're going to briefly introduce you to **LangGraph**, a popular multi-agent orchestration framework. We will specifically focus on implementing a **Conversational Tool-Calling** workflow, which allows an agent to reason verbally ("I should check the file...") before executing a tool.

### **Setup**

Before we begin, let's set up our environment by importing the necessary libraries and initializing our language model.


In [1]:
## =========================================================================
## SETUP: Environment Configuration and LLM Initialization
## =========================================================================
## This cell configures the NVIDIA API endpoint and initializes the LLM.
## We use ChatNVIDIA from langchain_nvidia_ai_endpoints for tool-capable models.

import requests
import nest_asyncio
from langchain_nvidia_ai_endpoints import ChatNVIDIA

## IMPORTANT: Apply nest_asyncio to prevent async event loop conflicts in Jupyter.
## This allows us to run async LangGraph code inside notebook cells without gridlock.
nest_asyncio.apply()

## Define model path and parameters
model_path = "http://llm_client:9000/v1"
model_name = requests.get(f"{model_path}/models").json().get("data", [{}])[0].get("id")

%env NVIDIA_BASE_URL=$model_path
%env NVIDIA_DEFAULT_MODE=open

## Override model name for local proxy setup
if "llm_client" in model_path:
    model_name = "meta/llama-3.3-70b-instruct"

## Initialize the LLM with tool-calling capabilities
## Note: temperature=0 ensures deterministic responses for reproducibility
llm = ChatNVIDIA(
    model=model_name, 
    base_url=model_path,
    max_completion_tokens=5000, 
    temperature=0, 
)

print(f"Initialized LLM: {model_name}")

env: NVIDIA_BASE_URL=http://llm_client:9000/v1
env: NVIDIA_DEFAULT_MODE=open
Initialized LLM: meta/llama-3.3-70b-instruct


----

And lastly, let's load in our notebook names and also the previously-computed notebook summaries dictionary. We'll default to just using the summaries throughout this notebook, but feel free to experiment. 

In [2]:
## =========================================================================
## DATA LOADING: Load Notebook Metadata and Summaries
## =========================================================================
## Load pre-computed notebook summaries to provide context for our agent.
## This gives the agent knowledge about course content without loading full notebooks.

import json
import os

## Load the notebook chunks/summaries from our pre-processed JSON file
with open('notebook_chunks.json', 'r') as fp:
    nbsummary = json.load(fp)

## Extract filenames for tool usage (so agent knows which notebooks exist)
filenames = nbsummary.get("filenames")

## Combine all outlines into a single context string for the system prompt
outlines = "\n\n".join([v.get("outline") for k,v in nbsummary.items() if isinstance(v, dict)])

print(f"Loaded {len(filenames)} notebook summaries")
# print(outlines)  # Uncomment to inspect the outlines

FileNotFoundError: [Errno 2] No such file or directory: 'notebook_chunks.json'

<hr>
<br>

## **Part 7.6.1:** Agentic Notebook Retrieval & The Conversational Tool Caller

In some cases, we may need an agent that can **Talk** and **Act**.
Native tool-calling (like `llm.bind_tools`) is efficient but often skips the "talking" part, jumping straight to code execution. To fix this, we will re-implement the **Conversational Tool Caller** pattern discussed in previous notebooks.

We will define a custom wrapper that:

1.  Injects tool definitions into the system prompt as an XML `<toolbank>`.
2.  Instructs the model to use `<function>` tags for actions.
3.  Parses the output to separate the **Conversational Text** from the **Structured Tool Call**.

In [3]:
## =========================================================================
## CONVERSATIONAL TOOL CALLER: Custom Wrapper for "Think-Then-Act" Pattern
## =========================================================================
## This class enables the LLM to verbally reason ("I should check the file...")
## BEFORE executing tool calls, rather than silently making tool calls.
##
## KEY INSIGHT: Native tool calling is efficient but lacks transparency.
## This wrapper injects tool definitions into the prompt and parses XML output,
## allowing the model to explain its reasoning while still producing structured calls.

import json
import re
from typing import List, Any, Dict
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.runnables import Runnable, RunnableLambda, RunnablePassthrough
from langchain_core.tools import BaseTool, tool
from langchain_core.utils.function_calling import convert_to_openai_tool

class ConversationalToolCaller:
    """
    A wrapper that uses prompt engineering to enable 'Conversational' tool calling.
    It encourages the model to 'Think' (generate text) before 'Acting' (generating XML tool calls).
    
    This is useful when you want:
    - Transparent reasoning before tool execution
    - User-friendly explanations of what the agent is doing
    - Control over the tool-calling format independent of the model's native capabilities
    """
    
    def __init__(self, llm: Runnable, tool_instruction: str):
        """Initialize with an LLM and custom tool instructions."""
        self.llm = llm
        self.tool_instruction = tool_instruction

    def bind_tools(self, tools: List[BaseTool]):
        """
        Simulates the .bind_tools() interface but uses Prompt Engineering + Output Parsing
        instead of native API tool binding.
        
        Returns a Runnable chain that:
        1. Adds tool definitions to the prompt
        2. Invokes the LLM
        3. Parses the output to extract tool calls from <function> tags
        """
        ## Step 1: Convert tools to OpenAI-format definitions for the prompt
        tool_defs = [convert_to_openai_tool(t) for t in tools]
        tool_json = json.dumps(tool_defs, indent=2)
        tool_inst = self.tool_instruction.format(toolbank=tool_json)
        
        ## Step 2: Define the Output Parser to extract tool calls from <function> tags
        def parse_output(message: BaseMessage) -> BaseMessage:
            """Parse <function> XML tags from the response and convert to tool_calls."""
            content = message.content
            tool_calls = []
            
            ## Handle incomplete closing tags (model stopped mid-generation)
            if "<function>" in content and "</function>" not in content:
                if content.rstrip().endswith("}"):
                    content += "</function>"
            
            ## Regex to find <function>...</function> blocks
            pattern = r'<function>(.*?)</function>'
            matches = list(re.finditer(pattern, content, re.DOTALL))
            
            if matches:
                ## Strip tool XML from content, keeping conversational text
                clean_content = re.sub(pattern, '', content).strip()
                
                ## Parse each function call into LangChain's tool_call format
                for match in matches:
                    try:
                        func_data = json.loads(match.group(1))
                        tool_calls.append({
                            "name": func_data.get("name"),
                            "args": func_data.get("parameters") or func_data.get("args", {}),
                            "id": f"call_{match.start()}",  # Generate unique ID
                            "type": "tool_call"
                        })
                    except json.JSONDecodeError:
                        ## Skip malformed JSON - model may have produced invalid syntax
                        pass
                
                ## Return AIMessage with both content AND structured tool_calls
                return AIMessage(content=clean_content, tool_calls=tool_calls)
            
            ## No tool calls found - return message unchanged
            return message

        def add_instructions(msgs):
            """Inject tool instructions into the system prompt."""
            ## Handle various input formats (list of messages, dict with messages, etc.)
            msgs = getattr(msgs, "messages", msgs)
            if isinstance(msgs, str):
                msgs = [("user", msgs)]
            if isinstance(msgs, tuple):
                msgs = [msgs]
            msgs = list(msgs)
            
            ## Append tool instructions to existing system message or create new one
            if len(msgs) > 0:
                first = msgs[0]
                if hasattr(first, "role") and first.role == "system":
                    first.content += "\n\n" + tool_inst
                elif isinstance(first, dict) and first.get("role") == "system":
                    msgs[0]["content"] += "\n\n" + tool_inst
                elif isinstance(first, tuple) and first[0] == "system":
                    msgs[0] = ("system", first[1] + "\n\n" + tool_inst)
                else:
                    ## No system message exists - prepend one
                    msgs = [("system", tool_inst)] + msgs
            else:
                msgs = [("system", tool_inst)]
            return msgs
            
        ## Step 3: Chain it all together using LangChain Expression Language (LCEL)
        ## Pipeline: add_instructions -> LLM (stop at </function>) -> parse_output
        return (
            RunnablePassthrough()
            | add_instructions
            | self.llm.bind(stop=["</function>"])  # Stop generating after tool call
            | parse_output
        )


## =========================================================================
## TOOL INSTRUCTION PROMPT: Tells the model HOW to use tools
## =========================================================================
## This prompt template is injected into the system message to teach the model
## the XML format for tool calls.

tool_instruction = (
    "In addition to your directive, you have access to the tools listed in the toolbank below."
    " You can invoke tools using the <function> tag."
    " You should ALWAYS reason about what you need to do before calling a tool."
    " \n\n<toolbank>\n{toolbank}\n</toolbank>\n"
    " \nTo call a tool, use this format:"
    ' \n<function>{{"name": "tool_name", "parameters": {{"param": "value"}}}}</function>'
)


## =========================================================================
## TEST: Verify the Conversational Tool Caller works
## =========================================================================

@tool
def multiply(a: float, b: float) -> float:
    """Multiplies a and b together and returns the result."""
    return a * b

## Initialize our Custom Wrapper
conv_llm = ConversationalToolCaller(llm, tool_instruction)

## Test it - notice how the response includes BOTH text AND tool_calls
result = conv_llm.bind_tools([multiply]).invoke(("user", "Please test out my tool. Follow the instructions as strictly as possible."))
print("Content:", result.content)
print("Tool Calls:", result.tool_calls)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Feel free to make modifications to these prompts as you go along, as there will be various tweaks you can make to improve your agent's performance.

<br>

### **Part 7.6.2:** Introducing LangGraph

The **[LangGraph framework](https://github.com/langchain-ai/langgraph)** is a new addition that allows us to manage the conversation flow using a state graph. By leveraging LangGraph, we can define the agent's states, transitions, and actions in a structured manner, eliminating the need for a fully-custom event loop. This framework enhances scalability and maintainability, especially when dealing with multi-agent systems or intricate workflows.

#### How LangGraph Enhances Our Workflow:
- **State Management:** LangGraph allows for clear delineation of different states within the conversation, making it easier to track and manage the agent's progress and decisions.
- **Conditional Transitions:** With LangGraph, we can define conditional edges that dictate how the conversation flows based on certain triggers or conditions.
- **Modularity:** The framework promotes modularity by allowing different nodes (functions) to handle specific tasks, facilitating easier updates and expansions.

#### Why Is LangGraph Better Than Custom?
- **Designed For Multi-Agent Systems:** Unlike our while-loop which we could massage into a workable multi-state system, LangGraph takes a state graph approach to modeling the agentic traversal process. As such, it incorporates design patterns which scale naturally to non-sequential and even dynamic routines.
- **Streamlined Integrations :** As a relatively popular framework, LangGraph has accumulated a plethora of free and premium integrations which can greatly improve the development and deployment experience. The development team has released integrations like LangServe, LangSmith, and LangGraph-Studio, and the community at large has contributed a variety of reference applications which showcase both domain-specific applications and modular plug-and-play components. If you want a reference example of a relatively-novel agentic paradigm, there's a decent chance somebody's making a reference implementation in LangGraph. 

#### When Is LangGraph Worse Than Custom?
- **Potential Overkill:** In order to account for various multi-agent-specific feature sets and edge cases, LangGraph implements some strong assumptions which greatly increase its learning curve. If you can implement your solutions in basic LangChain, the runnable paradigm is more than sufficient to streamline your pipeline and bypasses several layers of complexity introduced by LangGraph. If, on the other hand, you know you want to scale your application and can benefit from its well-thought-out features/examples, then perhaps it's worth diving in and getting comfortable.
- **Pidgeonholed Abstraction:** While LangGraph is amazing, there is still room beyond LangGraph for deeper optimizations and stronger modularization. Those looking to make highly-specialized microservices may be interested in custom multithreading/multiprocessing schemes, advanced graph algorithms, and advanced resource management strategies which LangGraph may not offer  For those interested, consider checking out [**Knowledge-Graph-RAG**](https://github.com/NVIDIA/GenerativeAIExamples/tree/main/community/knowledge_graph_rag) as a reasonable gateway into such topics.

----

For the rest of the notebook, we'll use LangGraph to manage the flow between an agent and its toolset in order to recreate our manual loop from before. While our application may not be complex enough to require it, getting practice with LangGraph is good to help kickstart your familiarity with the larger multi-agent ecosystem. 

Below, let's define a typical graph that connects a human input node with an agent response node:

In [6]:
## =========================================================================
## LANGGRAPH FUNDAMENTALS: State, Graph, and Node Definitions
## =========================================================================
## LangGraph uses a StateGraph to manage conversation flow.
## Key concepts:
##   - State: A TypedDict that holds the conversation data (messages, etc.)
##   - Nodes: Functions that process state and return updates
##   - Edges: Connections between nodes (can be conditional)
##   - Checkpointer: Saves state for memory/persistence across turns

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import AnyMessage, add_messages
from langchain_core.runnables import RunnableConfig
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated, Literal
from typing_extensions import TypedDict
from IPython.display import Image, display

##################################################################
## STATE DEFINITION
##################################################################
## The State class defines what data flows through our graph.
## `add_messages` is a reducer that appends new messages to the list
## rather than replacing it - critical for conversation history!

class State(TypedDict):
    """Graph state for message passing between nodes."""
    ## "add_messages" tells LangGraph to append new messages to the history
    messages: Annotated[list[AnyMessage], add_messages]
    ## "directives" tracks the original goal (optional, for complex workflows)
    directives: Annotated[list[AnyMessage], add_messages]


##################################################################
## GRAPH FACTORY FUNCTION
##################################################################
## This helper function creates a compiled LangGraph with:
##   - Custom nodes and edges
##   - Memory persistence (MemorySaver)
##   - Optional visualization

def create_graph(
    nodes, 
    edges, 
    conditional_edges=[], 
    state=State, 
    thread_id="42", 
    plot=True
):
    """
    Factory function to create a LangGraph application.
    
    Args:
        nodes: List of (name, function) tuples for graph nodes
        edges: List of (source, target) tuples for direct connections
        conditional_edges: List of (source, condition_fn, mapping) for branching
        state: The State class to use
        thread_id: Unique ID for conversation memory
        plot: Whether to display the graph visualization
    
    Returns:
        Tuple of (compiled_app, memory_saver, config_dict)
    """
    ## Create the StateGraph with our state schema
    graph = StateGraph(state)
    
    ## Add all nodes (each node is a function that processes state)
    for node_name, node_fn in nodes:
        graph.add_node(node_name, node_fn)
    
    ## Add direct edges (unconditional transitions)
    for source, target in edges:
        graph.add_edge(source, target)
    
    ## Add conditional edges (branching based on state)
    for cedge in conditional_edges:
        graph.add_conditional_edges(*cedge)
    
    ## Config for thread-based memory (enables conversation persistence)
    config = {"configurable": {"thread_id": thread_id}}
    
    ## MemorySaver provides in-memory checkpointing for conversation history
    memory = MemorySaver()
    
    ## Compile the graph with the checkpointer
    app = graph.compile(checkpointer=memory)

    ## Optionally visualize the graph structure
    if plot: 
        try:
            display(Image(app.get_graph(xray=True).draw_mermaid_png()))
        except Exception as e:
            print(f"Graph visualization skipped: {e}")
    
    return app, memory, config


##################################################################
## CHAT PROMPT TEMPLATE
##################################################################
## This template structures our conversation with:
##   - System prompt defining the agent's role
##   - Pre-loaded context (notebook outlines)
##   - Placeholder for dynamic message history

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful DLI Chatbot who can request and reason about notebooks."
        " Please help the user out by answering any of their questions."
    )),
    ("human", f"Here is the info I want you to work with: {outlines}"),
    ("ai", "Awesome! I will proceed with this understanding."),
    ## Placeholder for conversation history - gets filled dynamically
    ("placeholder", "{messages}") 
])


##################################################################
## NODE FUNCTIONS
##################################################################
## Each node function receives state and returns state updates.
## The returned dict is MERGED with existing state (not replaced).

def human_fn(state):
    """
    Human input node - gets user input and adds to message buffer.
    
    NOTE: In production, you'd get input from a UI/API rather than input().
    The returned message is appended to state['messages'] due to add_messages reducer.
    """
    user_input = input("[Human]: ")
    return {"messages": [("human", user_input)]}


async def assistant_fn(state, config: RunnableConfig, **kwargs):
    """
    Assistant node - prompts the LLM with message buffer and returns response.
    
    The config parameter is passed by LangGraph and contains:
      - thread_id for memory
      - callbacks for streaming/tracing
    """
    ## Create the chain: prompt template -> LLM
    chain = chat_prompt | llm
    
    ## Invoke asynchronously - config carries callbacks for streaming
    response = await chain.ainvoke(state, config)
    
    ## Return the response to be appended to messages
    return {"messages": [response]}


##################################################################
## CREATE THE BASIC CHAT GRAPH
##################################################################
## This creates a simple: START -> human -> assistant -> END flow

app, memory, config = create_graph(
    ## Nodes: functions that process state
    nodes = [
        ("assistant", assistant_fn), 
        ("human", human_fn)
    ],
    ## Edges: connections between nodes
    edges = [
        (START, "human"),       # Start with human input
        ("human", "assistant"), # Then assistant responds
        ("assistant", END),     # End after response
    ]
)

NameError: name 'outlines' is not defined

To stream from this compiled graph, you can *roughly* use it like a runnable with some extra configurations and options:

In [7]:
## =========================================================================
## BASIC GRAPH INVOCATION: Invoke and Stream Examples
## =========================================================================
## LangGraph apps support both synchronous invoke and async streaming.
## Use ainvoke() for single responses, astream() for token-by-token output.

## Method 1: Full invocation (waits for complete response)
output = await app.ainvoke({"messages": []}, config=config)
print("Full Response:")
print(output.get("messages")[-1].content)

print("\n" + "="*50 + "\n")

## Method 2: Streaming (token-by-token for real-time display)
## stream_mode="messages" yields individual message chunks
print("Streaming Response:")
async for msg, meta in app.astream({"messages": []}, stream_mode="messages", config=config):
    ## Only print content chunks, skip metadata-only updates
    if msg.content:
        print(msg.content, end="", flush=True)
print()

NameError: name 'app' is not defined

You can also use the following helper:

In [8]:
## =========================================================================
## STREAMING HELPER: Pretty-print streaming responses with metadata
## =========================================================================
## This helper function provides formatted streaming output with:
##   - Node labels (so you know which agent is responding)
##   - Tool call indicators
##   - Clean separation between responses

async def stream_response(app, config, **inputs):
    """
    Stream responses from a LangGraph app with formatted output.
    
    Args:
        app: Compiled LangGraph application
        config: Config dict with thread_id
        **inputs: Optional inputs (defaults to empty messages)
    """
    ## Default to empty messages if not provided
    if not inputs:
        inputs = {"messages": []}
    
    ## Track which nodes we've seen to print headers
    seen_nodes = set()
    
    ## Stream with message mode for token-by-token output
    async for msg, meta in app.astream(inputs, stream_mode="messages", config=config):
        ## Extract metadata about which node generated this message
        lg_node = meta.get("langgraph_node", "unknown")
        lg_checkpoint = meta.get("langgraph_checkpoint_ns", "")
        msg_type = type(msg).__name__
        
        ## Skip human input nodes (we already printed their input)
        if lg_node == "human": 
            continue
        
        ## Print content if present
        if msg.content:
            ## Print node header on first message from each node
            node_key = f"{lg_node}:{lg_checkpoint}"
            if node_key not in seen_nodes:
                print(f"\n[{lg_node.capitalize()}]: ", end="", flush=True)
                seen_nodes.add(node_key)
            
            ## For streaming chunks, print content directly
            if "Chunk" in msg_type:
                print(msg.content, end="", flush=True)
            else:
                ## For complete messages, show representation
                print(msg.content, end="", flush=True)
            
        ## Print tool calls if any (useful for debugging)
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print(f"\n[Tool Call]: {msg.tool_calls[0]}", flush=True)
    
    print("\n" + "-"*50)


## Test the streaming helper
## Try prompts like:
##   - "Hello World! What tools do you have?"
##   - "Hello! How's it going?"
##   - "Hello! Who are you?"
await stream_response(app, config)

NameError: name 'app' is not defined

**In your exploration, note the following features:**
- The compiled graph maintains conversational history! This is because we have a **checkpointer** (specified on graph compilation) working behind the scenes to keep track of dialog on thread "42" (dictated by our config). Note that the checkpointer is also more generally useful for integrations involving backtracking, archiving, human-in-the-loop, etc.
- You'll notice there is some decent buffer information in the metadata which could be good for output processing, filtering, archiving, etc.

<hr>
<br>

### **Part 7.6.3:** Recreating Our ReAct Loop

In the previous notebook, we implemented the basic ReAct loop using a custom buffer protocol in the form of a state dictionary, a while loop, and a break condition that triggers when a response doesn't invoke a tool. In LangGraph, this is a common paradigm often visualized with the following graph:

> <div><img src="imgs/lg_react.png" width="600"/></div>
>
> **Source: [ReAct Agent with Structured Output | LangGraph How-To Guides](https://langchain-ai.github.io/langgraph/how-tos/react-agent-structured-output/)**

Digging through resources like this, you will find multiple flavors of implementation which synergize the node logic, edge logic, and postprocessing logic to construct a cohesive streaming syseIn this exercise,


canwe will rebuild the ReAct (Reason + Act) loop using LangGraph and our `ConversationalToolCaller`.
Because our wrapper parses the XML into standard `tool_calls`, we can use LangGraph's standard logic to route the conversation!

1.  **Agent Node:** Uses `ConversationalToolCaller` to generate text + tool calls.
2.  **Tools Node:** Executes the tools.
3.  **Conditional Edge:** Routes based on whether `tool_calls` exist.

<!-- end list -->

In [9]:
## =========================================================================
## REACT LOOP: Reason + Act Pattern with LangGraph
## =========================================================================
## The ReAct pattern alternates between:
##   1. REASONING: Agent thinks about what to do
##   2. ACTING: Agent calls tools to get information
##   3. OBSERVING: Agent sees tool results
##   4. Repeat until task is complete
##
## LangGraph models this as a cycle: agent -> tools -> agent -> ...
## with a conditional edge to break out when no tools are needed.

from langgraph.prebuilt import ToolNode
from functools import partial

##################################################################
## STEP 1: Define a Test Tool
##################################################################

@tool
def multiply(a: float, b: float) -> float:
    """Multiplies two numbers together. Use this for multiplication calculations."""
    return float(a) * float(b)

tools = [multiply]


##################################################################
## STEP 2: Define the Agent Node
##################################################################
## The agent node uses our ConversationalToolCaller to:
##   - Generate reasoning text ("I need to multiply...")
##   - Produce structured tool calls in <function> tags
##   - Parse output into standard AIMessage with tool_calls

async def agent_fn(
    state, 
    config: RunnableConfig, 
    llm,           # The ConversationalToolCaller-wrapped LLM
    chat_prompt    # The prompt template with context
):
    """
    Agent node that decides what to do next.
    
    This node:
    1. Gets the current message history from state
    2. Invokes the LLM with tool capabilities
    3. Returns the response (which may contain tool_calls)
    """
    ## Get messages from state, applying the prompt template
    messages = chat_prompt.invoke(state).to_messages()
    
    ## Invoke the tool-capable LLM
    ## Output: AIMessage with content="reasoning..." and tool_calls=[...]
    response = await llm.ainvoke(messages, config)
    
    return {"messages": [response]}


##################################################################
## STEP 3: Define the Tools Node
##################################################################
## The ToolNode from langgraph.prebuilt automatically:
##   - Extracts tool_calls from the last message
##   - Executes each tool with the provided arguments
##   - Returns ToolMessage results
##
## We wrap it to add a "please continue" prompt so the agent
## knows to keep reasoning after seeing tool results.

def tools_fn(state, config: RunnableConfig, tool_node):
    """
    Tools node that executes tool calls and prompts continuation.
    
    After tool execution, we add a user message to prompt the agent
    to continue reasoning based on the tool results.
    """
    ## Execute all tool calls from the last message
    out = tool_node.invoke(state)
    
    ## Add a continuation prompt so agent processes the results
    out["messages"] = out.get("messages", []) + [
        HumanMessage(content="Please continue based on the tool results.")
    ]
    
    return out


##################################################################
## STEP 4: Define Routing Logic
##################################################################
## The conditional edge checks if the agent wants to use tools.
## If tool_calls exist -> route to "tools" node
## Otherwise -> route to END (task complete)

def loop_or_end(state) -> Literal["tools", "end"]:
    """
    Routing function for conditional edge.
    
    Returns:
        "tools" if the last message has tool calls
        "end" if the agent is done (no tool calls)
    """
    last_msg = state["messages"][-1]
    
    ## Check if the message has tool_calls attribute and it's non-empty
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    
    return "end"


##################################################################
## STEP 5: Build the ReAct Graph
##################################################################
## Graph structure:
##   START -> human -> agent <-> tools
##                        |
##                        v
##                       END

## Create the tool-equipped LLM using our ConversationalToolCaller
tooled_llm = conv_llm.bind_tools(tools)

## Create partial functions with dependencies injected
## This allows us to pass the LLM and tools to node functions
tooled_agent_fn = partial(agent_fn, llm=tooled_llm, chat_prompt=chat_prompt)
tooled_tools_fn = partial(tools_fn, tool_node=ToolNode(tools))

## Build the graph
app_react, _, config_react = create_graph(
    nodes = [
        ("human", human_fn),         # Get user input
        ("agent", tooled_agent_fn),  # LLM reasoning + tool calling
        ("tools", tooled_tools_fn),  # Execute tools
    ],
    edges = [
        (START, "human"),            # Start with user input
        ("human", "agent"),          # User -> Agent
        ("tools", "agent"),          # After tools, back to agent (loop!)
    ],
    conditional_edges = [
        ## From agent, check if we need tools or are done
        ("agent", loop_or_end, {"tools": "tools", "end": END})
    ],
    thread_id="react_demo"
)

NameError: name 'chat_prompt' is not defined

Test it out! Notice how the agent will "talk" before it "acts".

In [10]:
## =========================================================================
## TEST THE REACT AGENT: Math Questions with Tool Usage
## =========================================================================
## Try asking math questions that require the multiply tool.
## Notice how the agent:
##   1. THINKS: "I need to multiply these numbers..."
##   2. ACTS: Calls the multiply tool
##   3. OBSERVES: Sees the result
##   4. RESPONDS: Gives the final answer
##
## Example prompts:
##   - "Can you calculate 50 times 12? Explain your steps"
##   - "Can you calculate 5000 * 20 / 54.2. Only use tools (never work anything out by hand)!"

await stream_response(app_react, config_react)

NameError: name 'app_react' is not defined

<hr>
<br>

### **Part 7.6.4:** Equipping Our Agent

Given all of our building blocks, we can now make an initial tooled LLM agent without too much code. To exemplify a starter agent, we will provide one simple but powerful tool: `read_notebook`. This will allow the agent to enrich its context with the full content of a notebook on command.

In [11]:
## =========================================================================
## NOTEBOOK READER AGENT: Full Agent with Real Tool
## =========================================================================
## This agent can read actual course notebooks to answer questions.
## The read_notebook tool loads notebook content into the conversation,
## giving the agent access to detailed course material on demand.

from functools import partial
from typing import Literal
import json
import os

##################################################################
## UTILITY: Convert Jupyter Notebook to Markdown
##################################################################

def notebook_to_markdown(path: str) -> str:
    """
    Load a Jupyter notebook and convert it to readable Markdown format.
    
    This extracts:
    - Markdown cells as-is
    - Code cells wrapped in ```python blocks
    
    Args:
        path: Path to the .ipynb file
        
    Returns:
        Markdown-formatted string of notebook content
    """
    with open(path, 'r', encoding='utf-8') as file:
        notebook = json.load(file)
    
    markdown_content = []
    for cell in notebook['cells']:
        if cell['cell_type'] == 'code':
            ## Wrap code in markdown code block
            code = "".join(cell["source"])
            markdown_content.append(f'```python\n{code}\n```')
        elif cell['cell_type'] == 'markdown':
            ## Include markdown source directly
            markdown_content.append("".join(cell["source"]))
        ## Optionally include outputs (commented out for brevity)
        # for output in cell.get('outputs', []):
        #     if output['output_type'] == 'stream':
        #         markdown_content.append(f'```\n{"".join(output["text"])}\n```')
    
    return '\n\n'.join(markdown_content)


##################################################################
## TOOL: Read Notebook Content
##################################################################

@tool
def read_notebook(filename: str) -> str:
    """
    Displays a notebook file to yourself and the end-user.
    
    WARNING: These files are long, so only use this tool as a last resort
    when you need specific details from a notebook.
    
    Args:
        filename: The name of the notebook file to read
        
    Returns:
        The notebook content in Markdown format
    """
    return notebook_to_markdown(filename)

## ADVANCED: Modify the tool schema to constrain valid filenames
## This enables grammar-enforced generation on compatible servers,
## ensuring the model can only select from existing notebook files.
read_notebook.args_schema.model_json_schema()["properties"]["filename"]["enum"] = filenames


##################################################################
## OPTIONAL: Directive-Setting Entry Node
##################################################################
## This node allows setting persistent directives that guide the agent.
## Useful for multi-step tasks where the original goal should persist.

def set_directive_fn(state):
    """
    Entry node that captures the user's directive.
    
    Copies the initial user message to the 'directives' field
    so it persists even as the conversation grows.
    """
    ## Get the last human message as the directive
    messages = state.get("messages", [])
    if messages:
        return {"directives": [messages[-1]]}
    return {}


##################################################################
## BUILD THE FULL NOTEBOOK AGENT
##################################################################

toolset = [read_notebook]

## Create partial functions with the new toolset
tooled_agent_fn = partial(
    agent_fn, 
    llm=conv_llm.bind_tools(toolset), 
    chat_prompt=chat_prompt
)
tooled_tools_fn = partial(
    tools_fn, 
    tool_node=ToolNode(toolset)
)

## Build the graph with the notebook reader tool
app_final, memory_final, config_final = create_graph(
    nodes = [
        ("enter", set_directive_fn),   # Capture the user's goal
        ("agent", tooled_agent_fn),    # LLM with notebook reader
        ("tools", tooled_tools_fn),    # Execute read_notebook
    ],
    edges = [
        (START, "enter"),              # Start by capturing directive
        ("enter", "agent"),            # Then agent processes
        ("tools", "agent"),            # Loop: tools -> agent
    ],
    conditional_edges = [
        ("agent", loop_or_end, {"tools": "tools", "end": END})
    ],
    thread_id="notebook_agent",
    plot=True,
)

NameError: name 'filenames' is not defined

Now you have a fully agentic system! It uses **LangGraph** for orchestration and our custom **Conversational Tool Caller** to ensure high-quality, reasoned interactions.

In [12]:
## =========================================================================
## TEST THE NOTEBOOK AGENT
## =========================================================================
## Try asking questions that require reading specific notebooks.
## The agent will:
##   1. Reason about which notebook might have the answer
##   2. Call read_notebook to load the content
##   3. Extract and explain the relevant information
##
## Example prompts:
##   - "Show me an interesting code snippet from Notebook 5. Explain what it does."
##   - "How does the course explain embeddings?"
##   - "What topics are covered in the multimodal section?"

await stream_response(app_final, config_final, messages=[("user", "Show me an interesting code snippet from Notebook 5. Explain what it does. Abide by the format, and call the tool!")])

NameError: name 'app_final' is not defined

Note that this strategy on its own will lead to context overload without further measurements. Still, it is a solid step in the direction of agentics and offers good practice for building interesting state-managing applications.

<hr>
<br>

### **Part 7.6.5:** Continuing With LangGraph?

Going forward from this course, we hope you will be able to continue applying generative AI and agentic paradigms to make amazing and impactful systems! Though we didn't spend too much time with LangGraph, we encourage you to try out some of the [**tutorials**](https://academy.langchain.com/courses/intro-to-langgraph) and keep an eye out for interesting agentic paradigms as they come out!

<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>